In [ ]:
import json, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D

with open('Floor Plans/floor1_styled.json') as f:
    floor1s = json.load(f)

TYPE_COLORS = {
    'classroom':  '#D9E5F7',
    'lecture':    '#F5EDD6',
    'lobby':      '#FDF6E3',
    'corridor':   '#EEEEEE',
    'connection': '#E0F7FA',
    'door':       '#FFFFFF',
    'entrance':   '#FFFFFF',
    'lab':        '#DAF0DE',
    'library':    '#EDE7F6',
    'stairwell':  '#FFFFFF',
    'elevator':   '#CFD8DC',
    'office':     '#FFF3E0',
    'other':      '#F5F5F5',
}

W = floor1s['viewBox']['width']
H = floor1s['viewBox']['height']

fig, ax = plt.subplots(figsize=(20, 15))
ax.set_facecolor('#E8E0D0')
fig.patch.set_facecolor('#FFFFFF')

legend_done = set()

# Pass 1: draw all regular rooms first (zorder=2)
for room in floor1s['rooms']:
    if room.get('symbol', ''):
        continue
    pts = np.array(room['polygon'], dtype=float)
    pts[:, 1] = H - pts[:, 1]
    t = room['type']
    color = TYPE_COLORS.get(t, '#EEEEEE')
    cx, cy = pts[:, 0].mean(), pts[:, 1].mean()
    poly = Polygon(pts, closed=True, facecolor=color, edgecolor='#3C3C3C',
                   linewidth=1.0, zorder=2)
    ax.add_patch(poly)
    if room['name']:
        ax.text(cx, cy, room['name'], ha='center', va='center',
                fontsize=5, fontweight='bold', color='#222', zorder=3)
    if t not in legend_done:
        legend_done.add(t)

# Pass 2: draw stairwells on top (zorder=4) so they show through corridors
for room in floor1s['rooms']:
    if room.get('symbol', '') != 'hatch_diagonal':
        continue
    pts = np.array(room['polygon'], dtype=float)
    pts[:, 1] = H - pts[:, 1]
    bbox_w = pts[:, 0].max() - pts[:, 0].min()
    bbox_h = pts[:, 1].max() - pts[:, 1].min()
    hatch = '--' if bbox_h >= bbox_w else '||'
    poly = Polygon(pts, closed=True, facecolor='white', edgecolor='#3C3C3C',
                   hatch=hatch, linewidth=1.2, zorder=4)
    ax.add_patch(poly)
    t = room['type']
    if t not in legend_done:
        legend_done.add(t)

# Pass 3: draw doors on top of everything (zorder=5)
for room in floor1s['rooms']:
    if room.get('symbol', '') != 'door_arc':
        continue
    pts = np.array(room['polygon'], dtype=float)
    pts[:, 1] = H - pts[:, 1]
    poly = Polygon(pts, closed=True, facecolor='#4A4A4A', edgecolor='#222222',
                   linewidth=0.6, zorder=5)
    ax.add_patch(poly)
    t = room['type']
    if t not in legend_done:
        legend_done.add(t)

# Legend
legend_handles = []
for t in sorted(legend_done):
    if t == 'stairwell':
        legend_handles.append(
            mpatches.Patch(facecolor='white', edgecolor='#555',
                           hatch='--', label='Stairwell'))
    elif t in ('door', 'entrance', 'connection'):
        legend_handles.append(
            mpatches.Patch(facecolor='#4A4A4A', edgecolor='#222',
                           label=t.capitalize()))
    else:
        legend_handles.append(
            mpatches.Patch(facecolor=TYPE_COLORS.get(t, '#EEE'),
                           edgecolor='#555', label=t.capitalize()))

ax.legend(handles=legend_handles, loc='lower left',
          fontsize=7, framealpha=0.9,
          title='Room type', title_fontsize=8)

ax.set_xlim(0, W)
ax.set_ylim(0, H)
ax.set_aspect('equal')
ax.set_title('Harvard Science Center — Floor 1 (Styled)', fontsize=14,
             fontweight='bold', pad=12)
ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Rooms rendered: {len(floor1s["rooms"])}')


In [ ]:
# ── Navigation  ────────────────────────────────────────────────────────────────
# Set START and END to any room name shown in the list below.
# Names are case-insensitive and allow partial matches.

START = "R102"
END   = "Cabot Library"

# ── BFS ────────────────────────────────────────────────────────────────────────
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon, FancyArrowPatch
from collections import deque

with open('Floor Plans/floor1_styled.json') as f:
    floor1s = json.load(f)

rooms_by_id   = {r['id']: r for r in floor1s['rooms']}
rooms_by_name = {r['name'].lower(): r['id']
                 for r in floor1s['rooms'] if r.get('name')}

def resolve(query):
    """Return room id from name (case-insensitive, partial match) or id."""
    q = query.lower()
    if q in rooms_by_id:
        return q
    if q in rooms_by_name:
        return rooms_by_name[q]
    matches = [rid for name, rid in rooms_by_name.items() if q in name]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        names = [rooms_by_id[m]['name'] for m in matches]
        raise ValueError(f"Ambiguous: {names}")
    raise ValueError(f"Room not found: '{query}'")

def bfs(start_id, end_id):
    visited = {start_id}
    queue   = deque([[start_id]])
    while queue:
        path = queue.popleft()
        if path[-1] == end_id:
            return path
        for nb in rooms_by_id.get(path[-1], {}).get('neighbors', []):
            if nb not in visited and nb in rooms_by_id:
                visited.add(nb)
                queue.append(path + [nb])
    return None

start_id = resolve(START)
end_id   = resolve(END)
path_ids = bfs(start_id, end_id)

if path_ids is None:
    print(f"No path found from '{START}' to '{END}'")
else:
    path_names = [rooms_by_id[p]['name'] or p for p in path_ids]
    print(f"Path ({len(path_ids)} steps): " + "  →  ".join(path_names))

    # ── Render map with path ────────────────────────────────────────────────────
    W = floor1s['viewBox']['width']
    H = floor1s['viewBox']['height']

    TYPE_COLORS = {
        'classroom': '#D9E5F7', 'lecture':  '#F5EDD6', 'lobby':    '#FDF6E3',
        'corridor':  '#EEEEEE', 'connection':'#E0F7FA', 'door':     '#FFFFFF',
        'entrance':  '#FFFFFF', 'lab':      '#DAF0DE', 'library':  '#EDE7F6',
        'stairwell': '#FFFFFF', 'elevator': '#CFD8DC', 'office':   '#FFF3E0',
        'other':     '#F5F5F5',
    }
    PATH_COLOR  = '#FFB703'   # amber for intermediate rooms
    START_COLOR = '#2DC653'   # green
    END_COLOR   = '#E63946'   # red

    path_set = set(path_ids)

    fig, ax = plt.subplots(figsize=(20, 15))
    ax.set_facecolor('#E8E0D0')
    fig.patch.set_facecolor('#FFFFFF')

    # Pass 1 — regular rooms
    for room in floor1s['rooms']:
        if room.get('symbol', ''):
            continue
        pts = np.array(room['polygon'], dtype=float)
        pts[:, 1] = H - pts[:, 1]
        rid = room['id']

        if rid == start_id:
            fc = START_COLOR
        elif rid == end_id:
            fc = END_COLOR
        elif rid in path_set:
            fc = PATH_COLOR
        else:
            fc = TYPE_COLORS.get(room['type'], '#EEEEEE')

        ax.add_patch(Polygon(pts, closed=True, facecolor=fc,
                             edgecolor='#3C3C3C', linewidth=1.0, zorder=2))
        if room['name']:
            cx, cy = pts[:, 0].mean(), pts[:, 1].mean()
            weight = 'bold' if rid in path_set else 'normal'
            color  = '#000' if rid in path_set else '#555'
            ax.text(cx, cy, room['name'], ha='center', va='center',
                    fontsize=5, fontweight=weight, color=color, zorder=3)

    # Pass 2 — stairwells
    for room in floor1s['rooms']:
        if room.get('symbol', '') != 'hatch_diagonal':
            continue
        pts = np.array(room['polygon'], dtype=float)
        pts[:, 1] = H - pts[:, 1]
        bw = pts[:, 0].max() - pts[:, 0].min()
        bh = pts[:, 1].max() - pts[:, 1].min()
        hatch = '--' if bh >= bw else '||'
        fc = PATH_COLOR if room['id'] in path_set else 'white'
        ax.add_patch(Polygon(pts, closed=True, facecolor=fc,
                             edgecolor='#3C3C3C', hatch=hatch,
                             linewidth=1.2, zorder=4))

    # Pass 3 — doors
    for room in floor1s['rooms']:
        if room.get('symbol', '') != 'door_arc':
            continue
        pts = np.array(room['polygon'], dtype=float)
        pts[:, 1] = H - pts[:, 1]
        ax.add_patch(Polygon(pts, closed=True, facecolor='#4A4A4A',
                             edgecolor='#222222', linewidth=0.6, zorder=5))

    # Pass 4 — path line through labelAnchor centers
    path_xy = []
    for rid in path_ids:
        la = rooms_by_id[rid].get('labelAnchor', {})
        if la:
            path_xy.append((la['x'], H - la['y']))

    if len(path_xy) >= 2:
        xs = [p[0] for p in path_xy]
        ys = [p[1] for p in path_xy]
        ax.plot(xs, ys, color='#023E8A', linewidth=2.5,
                linestyle='--', zorder=6, alpha=0.85)
        # Arrow segments
        for i in range(len(path_xy) - 1):
            x0, y0 = path_xy[i]
            x1, y1 = path_xy[i + 1]
            mx, my = (x0 + x1) / 2, (y0 + y1) / 2
            ax.annotate('', xy=(mx + (x1-x0)*0.01, my + (y1-y0)*0.01),
                        xytext=(mx - (x1-x0)*0.01, my - (y1-y0)*0.01),
                        arrowprops=dict(arrowstyle='->', color='#023E8A',
                                        lw=2.0), zorder=7)
        # Step numbers
        for i, (x, y) in enumerate(path_xy):
            label = 'S' if i == 0 else ('E' if i == len(path_xy)-1 else str(i))
            ax.text(x, y, label, ha='center', va='center', fontsize=6,
                    fontweight='bold', color='white', zorder=9,
                    bbox=dict(boxstyle='circle,pad=0.25',
                              facecolor='#023E8A' if label not in ('S','E')
                              else (START_COLOR if label=='S' else END_COLOR),
                              edgecolor='white', linewidth=0.8))

    # Legend
    legend_entries = [
        mpatches.Patch(facecolor=START_COLOR, edgecolor='#333', label=f'Start: {rooms_by_id[start_id]["name"]}'),
        mpatches.Patch(facecolor=END_COLOR,   edgecolor='#333', label=f'End: {rooms_by_id[end_id]["name"]}'),
        mpatches.Patch(facecolor=PATH_COLOR,  edgecolor='#333', label='On route'),
        plt.Line2D([0],[0], color='#023E8A', linewidth=2, linestyle='--', label='Path'),
    ]
    ax.legend(handles=legend_entries, loc='lower left',
              fontsize=8, framealpha=0.92, title='Navigation', title_fontsize=9)

    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.set_aspect('equal')
    ax.set_title(f'Navigation: {rooms_by_id[start_id]["name"]}  →  {rooms_by_id[end_id]["name"]}',
                 fontsize=14, fontweight='bold', pad=12)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# ── Available rooms ─────────────────────────────────────────────────────────────
print("\nAvailable room names:")
nav_rooms = sorted(
    {r['name'] for r in floor1s['rooms']
     if r.get('name') and r.get('neighbors') and r['type'] not in ('door','entrance','connection')},
    key=lambda s: s.lower()
)
for i in range(0, len(nav_rooms), 4):
    print("  " + "   ".join(f"{n:<22}" for n in nav_rooms[i:i+4]))
